# ResNet50 Multi-Label Classification — Continue Training

**Purpose**: Load the pre-trained multi-label ResNet50 model and continue training from the saved checkpoint to improve accuracy further.

**Key Improvements**:
- Load best model from previous training
- Progressive backbone unfreezing
- Advanced augmentation techniques
- Multi-optimizer strategy (layer-wise LR)
- Focal Loss with class weighting
- AMP (Automatic Mixed Precision) for faster training
- Comprehensive validation metrics
- Early stopping with patience

## Cell 1: Install & Import Dependencies

In [ ]:
# Install required packages
!pip install -q torch torchvision pillow scikit-learn matplotlib seaborn tqdm

import torch
import torchvision
import numpy as np
import os
from pathlib import Path
import json
from datetime import datetime
import sys
import time

print("✓ Dependencies installed")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    torch.cuda.empty_cache()

## Cell 1b: Mount Google Drive

In [ ]:
# Mount Google Drive for Colab
from google.colab import drive
print("📂 Mounting Google Drive...")
drive.mount('/content/drive')
print("✓ Google Drive mounted")
print("\nYour Drive location: /content/drive/MyDrive")

## Cell 2: Configure Paths & Load Previous Training State

In [ ]:
# ===== CONFIGURATION FOR GOOGLE COLAB + DRIVE =====
print("🔧 Configuring paths for Google Drive...\\n")

# Google Drive paths
DRIVE_PATH = "/content/drive/MyDrive"
PROJECT_FOLDER = "smart-dental-ai"
BASE_DIR = os.path.join(DRIVE_PATH, PROJECT_FOLDER)
DATASET_DIR = os.path.join(BASE_DIR, "datasets")
CLASSIFICATION_DATASET = os.path.join(DATASET_DIR, "classification-dataset")

OUTPUT_DIR = os.path.join(BASE_DIR, "runs/classification-results")
MODEL_SAVE_DIR = os.path.join(OUTPUT_DIR, "models")

print(f"📁 Project directory: {BASE_DIR}")
print(f"📁 Dataset folder: {CLASSIFICATION_DATASET}")
print(f"📁 Model save dir: {MODEL_SAVE_DIR}\\n")

# Create output directories
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

# Verify paths exist
if not os.path.exists(CLASSIFICATION_DATASET):
    print(f"⚠️  WARNING: Dataset directory not found: {CLASSIFICATION_DATASET}")
    print("Please ensure you have uploaded smart-dental-ai folder to Google Drive")
else:
    print("✓ Paths configured successfully for Google Drive")

## Cell 3: Load Dataset & Define Classes

In [ ]:
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image as PILImage
import torch.nn as nn

print("📊 Setting up data loaders...\n")

IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2

# ===== MULTI-LABEL DATASET CLASS =====
class MultiLabelDentalDataset(Dataset):
    def __init__(self, root_dir, classes, transform=None, labels_json=None):
        self.root_dir = root_dir
        self.classes = classes
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        self.transform = transform
        self.samples = []

        multi_label_map = {}
        if labels_json and os.path.exists(labels_json):
            with open(labels_json, 'r') as f:
                multi_label_map = json.load(f)
            print(f"✓ Loaded multi-label annotations: {len(multi_label_map)} entries")

        for class_name in classes:
            class_dir = os.path.join(root_dir, class_name)
            if not os.path.isdir(class_dir):
                continue
            for fname in os.listdir(class_dir):
                fpath = os.path.join(class_dir, fname)
                if not os.path.isfile(fpath):
                    continue

                label_vec = torch.zeros(len(classes), dtype=torch.float32)
                if fname in multi_label_map:
                    for cname in multi_label_map[fname]:
                        if cname in self.class_to_idx:
                            label_vec[self.class_to_idx[cname]] = 1.0
                else:
                    label_vec[self.class_to_idx[class_name]] = 1.0

                self.samples.append((fpath, label_vec))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label_vec = self.samples[idx]
        image = PILImage.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label_vec


# ===== DATA AUGMENTATIONS =====
# Conservative augmentations for validation
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Aggressive augmentations for continued training
train_transform_aggressive = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(p=0.7),
    transforms.RandomVerticalFlip(p=0.4),
    transforms.RandomRotation(degrees=30),
    transforms.RandomAffine(degrees=0, translate=(0.15, 0.15), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.RandomPerspective(distortion_scale=0.3, p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.3, scale=(0.05, 0.2), ratio=(0.3, 3.0))  # Apply AFTER ToTensor
])

train_path = os.path.join(CLASSIFICATION_DATASET, 'train')
val_path = os.path.join(CLASSIFICATION_DATASET, 'val')

# Get class list
classes = sorted([d for d in os.listdir(train_path) if os.path.isdir(os.path.join(train_path, d))])
num_classes = len(classes)
print(f"\nClasses ({num_classes}): {classes}\n")

# Load datasets
train_labels_json = os.path.join(CLASSIFICATION_DATASET, 'train_labels.json')
val_labels_json = os.path.join(CLASSIFICATION_DATASET, 'val_labels.json')

train_dataset = MultiLabelDentalDataset(train_path, classes, train_transform_aggressive, train_labels_json)
val_dataset = MultiLabelDentalDataset(val_path, classes, val_transform, val_labels_json)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

print(f"✓ Data loaders created")
print(f"  Training samples : {len(train_dataset)}")
print(f"  Validation samples: {len(val_dataset)}")

## Cell 4: Load Pre-trained Model

In [ ]:
from torchvision.models import resnet50, ResNet50_Weights
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

print("🔄 Loading pre-trained ResNet50 model...\n")

# Load ResNet50 with ImageNet weights
weights = ResNet50_Weights.IMAGENET1K_V1
model = resnet50(weights=weights)
print("✓ ResNet50 loaded with ImageNet pretrained weights")

# Replace classification head
model.fc = nn.Sequential(
    nn.Linear(2048, 512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512, 128),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(128, num_classes)  # Raw logits
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"✓ Model moved to {device}")

# Attempt to load best model from previous training
best_model_path = os.path.join(MODEL_SAVE_DIR, 'best_model.pth')
checkpoint_path = os.path.join(MODEL_SAVE_DIR, 'resnet50_multilabel_dental.pth')

loaded_checkpoint = False
if os.path.exists(best_model_path):
    try:
        model.load_state_dict(torch.load(best_model_path, map_location=device))
        print(f"\n✓ Loaded best model from previous training: {best_model_path}")
        print(f"  File size: {os.path.getsize(best_model_path) / 1e6:.2f} MB")
        loaded_checkpoint = True
    except Exception as e:
        print(f"⚠️  Could not load {best_model_path}: {e}")

if not loaded_checkpoint and os.path.exists(checkpoint_path):
    try:
        model.load_state_dict(torch.load(checkpoint_path, map_location=device))
        print(f"\n✓ Loaded model from: {checkpoint_path}")
        print(f"  File size: {os.path.getsize(checkpoint_path) / 1e6:.2f} MB")
        loaded_checkpoint = True
    except Exception as e:
        print(f"⚠️  Could not load {checkpoint_path}: {e}")

if not loaded_checkpoint:
    print(f"\n⚠️  No pre-trained checkpoint found.")
    print(f"  Expected at: {best_model_path}")
    print(f"  Or at    : {checkpoint_path}")
    print(f"  Training will start from ImageNet weights.\n")

print(f"\n📊 Model Parameters:")
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"   Total parameters    : {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")

## Cell 5: Advanced Loss Functions

In [ ]:
# ===== FOCAL LOSS FOR CLASS IMBALANCE =====
class MultiLabelFocalLoss(nn.Module):
    """Focal Loss for multi-label classification addressing class imbalance."""
    def __init__(self, gamma=2.0, pos_weight=None, alpha=0.25):
        super().__init__()
        self.gamma = gamma
        self.pos_weight = pos_weight
        self.alpha = alpha

    def forward(self, inputs, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(
            inputs, targets, pos_weight=self.pos_weight, reduction='none')
        p_t = torch.exp(-bce)
        focal = (1 - p_t) ** self.gamma * bce
        return focal.mean()

# ===== COMPUTE CLASS WEIGHTS =====
print("📊 Computing class weights for imbalanced data...\n")
class_counts = {}
for class_name in classes:
    class_path = os.path.join(train_path, class_name)
    count = len([f for f in os.listdir(class_path) if os.path.isfile(os.path.join(class_path, f))])
    class_counts[class_name] = count

total_samples = sum(class_counts.values())
pos_weights = []
for class_name in classes:
    neg = total_samples - class_counts[class_name]
    pos = class_counts[class_name]
    weight = neg / (pos + 1e-8)
    pos_weights.append(weight)
    print(f"  {class_name:15} samples={pos:5}  weight={weight:.2f}")

pos_weight_tensor = torch.tensor(pos_weights, dtype=torch.float32).to(device)
print(f"\n✓ Class weights computed and moved to {device}")

## Cell 6: Progressive Layer Unfreezing Strategy

In [ ]:
def freeze_backbone(model):
    """Freeze all backbone parameters."""
    for name, param in model.named_parameters():
        if 'fc' not in name:
            param.requires_grad = False

def unfreeze_top_layers(model, num_blocks=1):
    """Progressively unfreeze top layers of backbone.
    ResNet50 has: layer4 (3 blocks), layer3 (6 blocks), layer2 (4 blocks), layer1 (3 blocks)
    """
    # Always unfreeze head
    for param in model.fc.parameters():
        param.requires_grad = True

    if num_blocks >= 1:
        for param in model.layer4.parameters():
            param.requires_grad = True
    if num_blocks >= 2:
        for param in model.layer3.parameters():
            param.requires_grad = True
    if num_blocks >= 3:
        for param in model.layer2.parameters():
            param.requires_grad = True
    if num_blocks >= 4:
        for param in model.layer1.parameters():
            param.requires_grad = True

def get_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("✓ Layer unfreezing utilities defined")
print(f"\nArchitecture overview:")
print(f"  - layer4: (3 blocks) - top layers")
print(f"  - layer3: (6 blocks)")
print(f"  - layer2: (4 blocks)")
print(f"  - layer1: (3 blocks) - bottom layers")
print(f"  - fc: classification head")

## Cell 7: Training utilities & Metrics

In [ ]:
from tqdm import tqdm
from sklearn.metrics import hamming_loss, accuracy_score, f1_score

def multilabel_accuracy(preds_logits, targets, threshold=0.5):
    """Exact match accuracy: all labels must match."""
    preds = (torch.sigmoid(preds_logits) > threshold).float()
    correct = (preds == targets).all(dim=1).float().mean()
    return correct.item() * 100

def hamming_loss_metric(preds_logits, targets, threshold=0.5):
    """Hamming loss: fraction of labels that are incorrectly predicted."""
    preds = (torch.sigmoid(preds_logits) > threshold).float()
    return hamming_loss(targets.cpu().numpy(), preds.cpu().numpy())

def subset_accuracy(preds_logits, targets, threshold=0.5):
    """Subset accuracy: 0/1 loss."""
    preds = (torch.sigmoid(preds_logits) > threshold).float()
    return (preds == targets).all(dim=1).float().mean().item() * 100

print("✓ Metrics defined:")
print(f"  - Exact Match Accuracy")
print(f"  - Hamming Loss")
print(f"  - Subset Accuracy")

## Cell 8: Phase 1 - Head Fine-tuning with Focal Loss

In [ ]:
print("🚀 Phase 1: Head Fine-tuning with Focal Loss\n")
print("Configuration:")
print(f"  - Backbone: FROZEN")
print(f"  - Head: TRAINABLE")
print(f"  - Loss: Focal Loss with class weights")
print(f"  - LR: 0.0005")
print(f"  - Epochs: 10\n")

freeze_backbone(model)
criterion_phase1 = MultiLabelFocalLoss(gamma=2.0, pos_weight=pos_weight_tensor, alpha=0.25)
optimizer_phase1 = optim.AdamW(model.fc.parameters(), lr=0.0005, weight_decay=1e-5)
scheduler_phase1 = CosineAnnealingWarmRestarts(optimizer_phase1, T_0=3, T_mult=2, eta_min=1e-6)

PHASE1_EPOCHS = 10
best_val_acc_phase1 = 0.0
best_model_phase1_path = os.path.join(MODEL_SAVE_DIR, 'phase1_best.pth')
patience_phase1 = 0
PATIENCE_LIMIT = 5

phase1_history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(1, PHASE1_EPOCHS + 1):
    # Training
    model.train()
    train_loss, train_acc = 0.0, 0.0
    pbar = tqdm(train_loader, desc=f"Phase1 Epoch {epoch}/{PHASE1_EPOCHS}", leave=True)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer_phase1.zero_grad()
        outputs = model(images)
        loss = criterion_phase1(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.fc.parameters(), max_norm=1.0)
        optimizer_phase1.step()
        train_loss += loss.item()
        train_acc += multilabel_accuracy(outputs, labels)
        pbar.set_postfix({'Loss': f'{train_loss/(pbar.n+1):.4f}', 'Acc': f'{train_acc/(pbar.n+1):.2f}%'})

    # Validation
    model.eval()
    val_loss, val_acc = 0.0, 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion_phase1(outputs, labels)
            val_loss += loss.item()
            val_acc += multilabel_accuracy(outputs, labels)

    train_loss /= len(train_loader)
    train_acc /= len(train_loader)
    val_loss /= len(val_loader)
    val_acc /= len(val_loader)

    phase1_history['train_loss'].append(train_loss)
    phase1_history['train_acc'].append(train_acc)
    phase1_history['val_loss'].append(val_loss)
    phase1_history['val_acc'].append(val_acc)

    scheduler_phase1.step()

    print(f"Epoch [{epoch:02d}/{PHASE1_EPOCHS}] "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}%")

    if val_acc > best_val_acc_phase1:
        best_val_acc_phase1 = val_acc
        patience_phase1 = 0
        torch.save(model.state_dict(), best_model_phase1_path)
        print(f"  ✓ Best model saved (Acc: {val_acc:.2f}%)")
    else:
        patience_phase1 += 1
        if patience_phase1 >= PATIENCE_LIMIT:
            print(f"  ⚠️  Patience limit reached. Early stopping triggered.")
            break

print(f"\n✅ Phase 1 Complete. Best Val Accuracy: {best_val_acc_phase1:.2f}%")

## Cell 9: Phase 2 - Progressive Fine-tuning with Backbone Unfreezing

In [ ]:
print("\n🚀 Phase 2: Progressive Backbone Fine-tuning\n")
print("Configuration:")
print(f"  - Backbone: PROGRESSIVELY UNFROZEN (layer4 + layer3)")
print(f"  - Head: TRAINABLE")
print(f"  - Loss: Focal Loss with class weights")
print(f"  - LR: 0.00001 (backbone), 0.0001 (head)")
print(f"  - Epochs: 15\n")

# Load best model from Phase 1
if os.path.exists(best_model_phase1_path):
    model.load_state_dict(torch.load(best_model_phase1_path, map_location=device))
    print(f"✓ Loaded best model from Phase 1: Acc {best_val_acc_phase1:.2f}%\n")

# Unfreeze top layers for fine-tuning
unfreeze_top_layers(model, num_blocks=2)  # layer4 + layer3
trainable = get_trainable_params(model)
print(f"✓ Unfroze layer4 and layer3")
print(f"  Trainable parameters: {trainable:,}\n")

# Create layer-wise learning rates
layer_groups = [
    {'params': model.layer4.parameters(), 'lr': 0.00002},
    {'params': model.layer3.parameters(), 'lr': 0.00001},
    {'params': model.fc.parameters(), 'lr': 0.0001}
]

criterion_phase2 = MultiLabelFocalLoss(gamma=2.5, pos_weight=pos_weight_tensor, alpha=0.25)
optimizer_phase2 = optim.AdamW(layer_groups, weight_decay=2e-5)
scheduler_phase2 = CosineAnnealingWarmRestarts(optimizer_phase2, T_0=4, T_mult=2, eta_min=1e-7)

PHASE2_EPOCHS = 15
best_val_acc_phase2 = best_val_acc_phase1
best_model_phase2_path = os.path.join(MODEL_SAVE_DIR, 'phase2_best.pth')
patience_phase2 = 0
PATIENCE_LIMIT_P2 = 8

phase2_history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(1, PHASE2_EPOCHS + 1):
    # Training
    model.train()
    train_loss, train_acc = 0.0, 0.0
    pbar = tqdm(train_loader, desc=f"Phase2 Epoch {epoch}/{PHASE2_EPOCHS}", leave=True)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer_phase2.zero_grad()
        outputs = model(images)
        loss = criterion_phase2(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer_phase2.step()
        train_loss += loss.item()
        train_acc += multilabel_accuracy(outputs, labels)
        pbar.set_postfix({'Loss': f'{train_loss/(pbar.n+1):.4f}', 'Acc': f'{train_acc/(pbar.n+1):.2f}%'})

    # Validation
    model.eval()
    val_loss, val_acc = 0.0, 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion_phase2(outputs, labels)
            val_loss += loss.item()
            val_acc += multilabel_accuracy(outputs, labels)

    train_loss /= len(train_loader)
    train_acc /= len(train_loader)
    val_loss /= len(val_loader)
    val_acc /= len(val_loader)

    phase2_history['train_loss'].append(train_loss)
    phase2_history['train_acc'].append(train_acc)
    phase2_history['val_loss'].append(val_loss)
    phase2_history['val_acc'].append(val_acc)

    scheduler_phase2.step()

    improvement = val_acc - best_val_acc_phase2
    print(f"Epoch [{epoch:02d}/{PHASE2_EPOCHS}] "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}%", end="")

    if val_acc > best_val_acc_phase2:
        best_val_acc_phase2 = val_acc
        patience_phase2 = 0
        torch.save(model.state_dict(), best_model_phase2_path)
        print(f" | ✓ {improvement:+.2f}%")
    else:
        patience_phase2 += 1
        if patience_phase2 >= PATIENCE_LIMIT_P2:
            print(f"\n  ⚠️  Patience limit reached. Early stopping triggered.")
            # Load best model
            model.load_state_dict(torch.load(best_model_phase2_path, map_location=device))
            break
        print(f" | Patience: {patience_phase2}/{PATIENCE_LIMIT_P2}")

print(f"\n✅ Phase 2 Complete. Best Val Accuracy: {best_val_acc_phase2:.2f}%")
print(f"   Improvement from Phase 1: {best_val_acc_phase2 - best_val_acc_phase1:+.2f}%")

## Cell 10: Save Final Model

In [ ]:
# Load best overall model
final_model_path = os.path.join(MODEL_SAVE_DIR, 'best_model.pth')
if best_val_acc_phase2 > best_val_acc_phase1:
    model.load_state_dict(torch.load(best_model_phase2_path, map_location=device))
    print(f"Using Phase 2 model (Acc: {best_val_acc_phase2:.2f}%)")
else:
    if os.path.exists(best_model_phase1_path):
        model.load_state_dict(torch.load(best_model_phase1_path, map_location=device))
        print(f"Using Phase 1 model (Acc: {best_val_acc_phase1:.2f}%)")

## Cell 11: Comprehensive Validation Metrics

In [ ]:
from sklearn.metrics import classification_report, precision_recall_fscore_support
import numpy as np

print("📊 Comprehensive Validation Analysis\n")
print("="*70)

model.eval()
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels in tqdm(val_loader, desc="Evaluating"):
        images = images.to(device)
        outputs = model(images)
        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).float()
        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.numpy())
        all_probs.append(probs.cpu().numpy())

all_preds = np.vstack(all_preds)
all_labels = np.vstack(all_labels)
all_probs = np.vstack(all_probs)

print("\n1️⃣  GLOBAL METRICS:")
print("-" * 70)
exact_match = (all_preds == all_labels).all(axis=1).mean() * 100
hamming = hamming_loss(all_labels, all_preds)
print(f"   Exact Match Accuracy: {exact_match:.2f}%")
print(f"   Hamming Loss        : {hamming:.4f}")

print("\n2️⃣  PER-CLASS METRICS:")
print("-" * 70)
print(f"{'Class':<18} {'Precision':<12} {'Recall':<12} {'F1-Score':<12}")
print("-" * 70)

for i, cls in enumerate(classes):
    tp = ((all_preds[:, i] == 1) & (all_labels[:, i] == 1)).sum()
    fp = ((all_preds[:, i] == 1) & (all_labels[:, i] == 0)).sum()
    fn = ((all_preds[:, i] == 0) & (all_labels[:, i] == 1)).sum()
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * (precision * recall) / (precision + recall + 1e-8)
    print(f"{cls:<18} {precision:<12.4f} {recall:<12.4f} {f1:<12.4f}")

print("=" * 70)

## Cell 12: Plot Training Progress

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("darkgrid")
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Phase 1 Loss
axes[0, 0].plot(phase1_history['train_loss'], label='Train Loss', linewidth=2, marker='o')
axes[0, 0].plot(phase1_history['val_loss'], label='Val Loss', linewidth=2, marker='s')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Phase 1: Head Fine-tuning - Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Phase 1 Accuracy
axes[0, 1].plot(phase1_history['train_acc'], label='Train Acc', linewidth=2, marker='o')
axes[0, 1].plot(phase1_history['val_acc'], label='Val Acc', linewidth=2, marker='s')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy (%)')
axes[0, 1].set_title('Phase 1: Head Fine-tuning - Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Phase 2 Loss
axes[1, 0].plot(phase2_history['train_loss'], label='Train Loss', linewidth=2, marker='o')
axes[1, 0].plot(phase2_history['val_loss'], label='Val Loss', linewidth=2, marker='s')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].set_title('Phase 2: Backbone Fine-tuning - Loss')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Phase 2 Accuracy
axes[1, 1].plot(phase2_history['train_acc'], label='Train Acc', linewidth=2, marker='o')
axes[1, 1].plot(phase2_history['val_acc'], label='Val Acc', linewidth=2, marker='s')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Accuracy (%)')
axes[1, 1].set_title('Phase 2: Backbone Fine-tuning - Accuracy')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'continued_training_history.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Plot saved to: {os.path.join(OUTPUT_DIR, 'continued_training_history.png')}")

## Cell 13: Inference - Prediction on New Images